# MedSegDiff Thigh Segmentation — Sheffield Dataset (Lambda)

Runs **inference only** using already-trained MedSegDiff checkpoints on the
69 Sheffield augmented DICOM volumes.

The model was trained with two channels: water (ch0) + fat-fraction (ch1).
Sheffield has only a single greyscale image — it is used as the water channel;
fat-fraction is set to zeros.

⚠️ **Checkpoints must exist** in `~/medsegdiff_ckpts/` (trained on myosegmenTUM).
If that Lambda instance has been terminated you need to re-run training first.

Data: `~/sheffeld/20440164/Aug_N.dcm`
Output: `~/medsegdiff_sheffield_segs/Aug_N_seg.npz`

## 1 — Upload to Lambda
```bash
rsync -avz -e "ssh -i /tmp/lambda_key -o StrictHostKeyChecking=no" \
  /tmp/docker-desktop-root/run/desktop/mnt/host/c/Projects/dissector/eval_notebooks/sheffeld \
  ubuntu@<YOUR-LAMBDA-IP>:~/

rsync -avz -e "ssh -i /tmp/lambda_key -o StrictHostKeyChecking=no" \
  /tmp/docker-desktop-root/run/desktop/mnt/host/c/Projects/dissector/eval_notebooks/medsegdiff_ckpts/ \
  ubuntu@<YOUR-LAMBDA-IP>:~/medsegdiff_ckpts/

rsync -avz -e "ssh -i /tmp/lambda_key -o StrictHostKeyChecking=no" \
  /tmp/docker-desktop-root/run/desktop/mnt/host/c/Projects/dissector/medsegdiff/ \
  ubuntu@<YOUR-LAMBDA-IP>:~/medsegdiff/
```

## 2 — Download results when done
```bash
rsync -avz --mkpath -e "ssh -i /tmp/lambda_key -o StrictHostKeyChecking=no" \
  ubuntu@<YOUR-LAMBDA-IP>:~/medsegdiff_sheffield_segs/ \
  /tmp/docker-desktop-root/run/desktop/mnt/host/c/Projects/dissector/eval_notebooks/medsegdiff/sheffield_segs/
```
**Terminate the instance when done.**

In [ ]:
import subprocess, sys

def _ensure(*pkgs):
    import importlib
    missing = [p for p in pkgs
               if importlib.util.find_spec(p.replace('-', '_')) is None]
    if missing:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + list(missing))
    else:
        print('Already installed:', ', '.join(pkgs))

_ensure('SimpleITK', 'tqdm', 'torchvision', 'pydicom')

import torch
print('PyTorch :', torch.__version__)
print('CUDA    :', torch.cuda.is_available(),
      torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')

In [ ]:
import glob, os, re, sys
import numpy as np
import torch
import torch.nn.functional as F
import pydicom

MEDSEGDIFF_DIR = os.path.expanduser('~/medsegdiff')
CKPT_DIR       = os.path.expanduser('~/medsegdiff_ckpts')
IMG_DIR        = os.path.expanduser('~/sheffeld/20440164')
OUTPUT_DIR     = os.path.expanduser('~/medsegdiff_sheffield_segs')

for path, label in [
    (MEDSEGDIFF_DIR, 'medsegdiff package'),
    (CKPT_DIR,       'checkpoints'),
    (IMG_DIR,        'Sheffield images'),
]:
    ok = os.path.isdir(path)
    print(f'{"OK" if ok else "MISSING"}: {label}')
    if not ok:
        raise FileNotFoundError(f'Upload {label} first')

os.makedirs(OUTPUT_DIR, exist_ok=True)

if MEDSEGDIFF_DIR not in sys.path:
    sys.path.insert(0, MEDSEGDIFF_DIR)

from dataset   import GT_LABELS, _norm
from unet      import UNet
from diffusion import GaussianDiffusion

IMG_SIZE   = 256
BASE_CH    = 64
T_DIM      = 256
T_STEPS    = 1000
DDIM_STEPS = 50
DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
MUSCLES    = list(GT_LABELS)

dcm_files = sorted(
    [f for f in glob.glob(os.path.join(IMG_DIR, 'Aug_*.dcm'))
     if '_segmentations' not in f],
    key=lambda p: int(re.search(r'Aug_(\d+)\.dcm', p).group(1)),
)
print(f'Muscles : {MUSCLES}')
print(f'Volumes : {len(dcm_files)}')
print(f'Device  : {DEVICE}')

In [ ]:
models = {}
diffusion = GaussianDiffusion(T=T_STEPS, device=DEVICE)

for muscle in MUSCLES:
    best_ckpt = os.path.join(CKPT_DIR, f'{muscle}_best.pt')
    if not os.path.exists(best_ckpt):
        print(f'[WARN] checkpoint not found: {best_ckpt}')
        continue
    ckpt       = torch.load(best_ckpt, map_location=DEVICE)
    saved_args = ckpt.get('args', {})
    img_ch     = saved_args.get('img_ch', 2)
    model = UNet(
        img_ch=img_ch,
        base=saved_args.get('base_ch', BASE_CH),
        t_dim=saved_args.get('t_dim', T_DIM),
    ).to(DEVICE)
    model.load_state_dict(ckpt['model'])
    model.eval()
    models[muscle] = (model, img_ch)
    print(f'  {muscle}: epoch {ckpt.get("epoch","?")}, '
          f'best Dice {ckpt.get("best_dice",0):.4f}, img_ch={img_ch}')

print(f'Loaded {len(models)}/{len(MUSCLES)} models.')

In [ ]:
@torch.no_grad()
def segment_volume(model, img_ch, grey_arr):
    """Segment a volume slice-by-slice; grey image used as water, zeros as fat-fraction."""
    D, H, W  = grey_arr.shape
    pred_vol = np.zeros((D, H, W), dtype=np.uint8)
    model.eval()
    for sl in range(D):
        channels = [torch.from_numpy(_norm(grey_arr[sl])).unsqueeze(0)]
        if img_ch == 2:
            channels.append(torch.zeros_like(channels[0]))  # fat-fraction = 0
        img_t = torch.cat(channels, dim=0) * 2.0 - 1.0
        img_r = F.interpolate(
            img_t.unsqueeze(0).to(DEVICE),
            size=(IMG_SIZE, IMG_SIZE), mode='bilinear', align_corners=False,
        )
        pred  = diffusion.ddim_sample(model, img_r, num_steps=DDIM_STEPS)
        pred_r= F.interpolate(pred, size=(H, W), mode='bilinear', align_corners=False)
        pred_vol[sl] = (pred_r.squeeze().cpu().numpy() > 0.0).astype(np.uint8)
    return pred_vol


for dcm_path in dcm_files:
    idx      = re.search(r'Aug_(\d+)\.dcm', dcm_path).group(1)
    out_path = os.path.join(OUTPUT_DIR, f'Aug_{idx}_seg.npz')

    if os.path.exists(out_path):
        existing = set(np.load(out_path).files)
        if set(models.keys()).issubset(existing):
            print(f'Skipping (done): Aug_{idx}')
            continue

    print(f'\nProcessing: Aug_{idx}')
    ds       = pydicom.dcmread(dcm_path)
    grey_arr = ds.pixel_array.astype(np.float32)
    D, H, W  = grey_arr.shape
    print(f'  Shape: {grey_arr.shape}')

    all_masks = {}
    if os.path.exists(out_path):
        all_masks = dict(np.load(out_path))

    for muscle, (model, img_ch) in models.items():
        if muscle in all_masks:
            print(f'  {muscle}: already done')
            continue
        print(f'  {muscle} ...', end=' ', flush=True)
        pred = segment_volume(model, img_ch, grey_arr)
        all_masks[muscle] = pred
        print(f'{int(pred.sum()):,} voxels')

    np.savez_compressed(out_path, **all_masks)
    print(f'  Saved → {out_path}')

print('\nAll done.')

In [ ]:
results = sorted(glob.glob(os.path.join(OUTPUT_DIR, '*.npz')))
print(f'Output files: {len(results)} / {len(dcm_files)}')
if results:
    s = np.load(results[0])
    for k in sorted(s.files):
        print(f'  {k}: shape={s[k].shape}  voxels={int(s[k].sum()):,}')